# 01 — Download & Inspect Sen1Floods11 (Hand-Labeled Subset)

This notebook downloads **only** the hand-labeled Sentinel-1 flood subset (~446 chips, avoiding the ~14 GB full bucket) from `gs://sen1floods11/v1.1`:
- Split CSVs: `flood_train_data.csv` (252), `flood_valid_data.csv` (89), `flood_test_data.csv` (90), `flood_bolivia_data.csv` (15)
- Sentinel-1 chips (`S1Hand/`): 2 bands (`VV`, `VH`) in dB, 512x512, EPSG:4326, 10 m
- Hand labels (`LabelHand/`): `-1` no data, `0` not water, `1` water

**Citation:** Bonafilia et al. (2020), *Sen1Floods11: A Georeferenced Dataset to Train and Test Deep Learning Flood Algorithms for Sentinel-1*, CVPR Workshops.

In [ ]:
# Colab setup: clone repo or set working directory to repo root
import os, sys
from pathlib import Path

if Path('/content/geoterra-ai').exists():
    %cd /content/geoterra-ai
REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Working directory:', REPO_ROOT)

In [ ]:
# Install dependencies if running on Colab
# !pip install -q -r ml/requirements.txt

In [ ]:
from ml import config

config.SPLITS_DIR.mkdir(parents=True, exist_ok=True)
config.S1_HAND_DIR.mkdir(parents=True, exist_ok=True)
config.LABEL_HAND_DIR.mkdir(parents=True, exist_ok=True)
print('Target directories ready under:', config.SEN1FLOODS11_DIR)

In [ ]:
# Download ONLY the 4 official split CSVs and HandLabeled S1 + Label chips via gsutil
!gsutil -m cp \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_train_data.csv \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_valid_data.csv \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_test_data.csv \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_bolivia_data.csv \
  ml/data/sen1floods11/splits/

!gsutil -m rsync -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand ml/data/sen1floods11/S1Hand/
!gsutil -m rsync -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand ml/data/sen1floods11/LabelHand/

In [ ]:
# Verify split counts and inspect sample VV, VH, and hand-label chips
import numpy as np
import matplotlib.pyplot as plt
from ml.src.dataset import read_split_csv, load_s1_chip, load_label_chip, compute_channel_stats

splits = {
    name: read_split_csv(config.SPLITS_DIR / fname)
    for name, fname in [
        ('train', 'flood_train_data.csv'),
        ('val', 'flood_valid_data.csv'),
        ('test', 'flood_test_data.csv'),
        ('bolivia', 'flood_bolivia_data.csv'),
    ]
}
for k, v in splits.items():
    print(f'{k:8s}: {len(v)} chips')

train_stats = compute_channel_stats(splits['train'])
print('Train-split VV/VH dB stats:', train_stats)

sample_s1, sample_lbl = splits['train'][0]
arr = load_s1_chip(config.S1_HAND_DIR / sample_s1)
lbl = load_label_chip(config.LABEL_HAND_DIR / sample_lbl)
print(f'Sample {sample_s1}: shape={arr.shape}, min={arr.min():.2f} dB, max={arr.max():.2f} dB, labels={np.unique(lbl)}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(arr[0], cmap='gray', vmin=-25, vmax=0); axes[0].set_title('VV (dB)')
axes[1].imshow(arr[1], cmap='gray', vmin=-30, vmax=-5); axes[1].set_title('VH (dB)')
axes[2].imshow(lbl, cmap='viridis', vmin=-1, vmax=1); axes[2].set_title('Hand Label (-1/0/1)')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()